# Warehouse Liquidation Strategy

This notebook builds the liquidation lists and final report from the inventory data and merchandising memo.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.optimize import milp, LinearConstraint, Bounds
from pathlib import Path

# Submission version: use the platform workspace directly.
WORKSPACE_DIR = Path("/workspace")
DATA_DIR = WORKSPACE_DIR / "data"

DATA_PATH = DATA_DIR / "Q3_Inventory_Sales_Data.csv"
MEMO_PATH = DATA_DIR / "merchandising_strategy.txt"

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)

ANCHOR_DATE = pd.Timestamp('2024-09-26')
RECENCY_CUTOFF = ANCHOR_DATE - pd.Timedelta(weeks=8)
SALES_COLUMNS = [f'Week_{i}' for i in range(1, 13)]


AttributeError: module 'matplotlib' has no attribute 'get_data_path'

In [3]:
with open(MEMO_PATH, "r", encoding="utf-8") as f:
    memo_text = f.read()

print(memo_text)

df = pd.read_csv(DATA_PATH)
df["Launch_Date"] = pd.to_datetime(df["Launch_Date"])

print("Shape:", df.shape)
display(df.head(8))

NameError: name 'MEMO_PATH' is not defined

## Normalize sales and apply business rules

In [4]:
def normalized_avg_weekly_sales(row: pd.Series) -> float:
    sales = row[SALES_COLUMNS].to_numpy(dtype=float)

    weeks_since_launch = max(1, (ANCHOR_DATE - row["Launch_Date"]).days // 7)
    weeks_active = min(12, weeks_since_launch)

    active_sales = sales[-weeks_active:]

    if len(active_sales) > 1:
        stdev = np.std(active_sales, ddof=1)
        if stdev > 0:
            z_scores = np.abs((active_sales - np.mean(active_sales)) / stdev)
            anomaly_mask = z_scores >= 2.5
            if anomaly_mask.any():
                non_anomalous = active_sales[~anomaly_mask]
                if len(non_anomalous):
                    replacement_value = np.median(non_anomalous)
                else:
                    replacement_value = np.median(active_sales)
                active_sales = np.where(anomaly_mask, replacement_value, active_sales)

    return float(np.mean(active_sales))

df["avg_weekly_sales"] = df.apply(normalized_avg_weekly_sales, axis=1)
df["is_damaged"] = df["Condition"].eq("Damaged")
df["is_dead"] = (~df["is_damaged"]) & (((df["Inventory_Qty"] / np.maximum(df["avg_weekly_sales"], 0.1)) * 7) > 730)
df["is_protected"] = (df["Category"].eq("Home Automation")) | (df["Launch_Date"] >= RECENCY_CUTOFF)

df["effective_margin"] = np.where(df["is_damaged"] | df["is_dead"], 0.0, df["Inventory_Qty"] * df["Unit_Margin"])
df["effective_volume"] = df["Inventory_Qty"] * df["Unit_Volume"]

print("Total rows:", len(df))
print("Protected rows:", int(df["is_protected"].sum()))
print("Eligible rows:", int((~df["is_protected"]).sum()))
print("Damaged rows:", int(df["is_damaged"].sum()))
print("Dead rows:", int(df["is_dead"].sum()))
display(df[["SKU", "Category", "Condition", "Launch_Date", "avg_weekly_sales", "is_damaged", "is_dead", "is_protected"]].head(12))

NameError: name 'df' is not defined

## Build the constrained optimization model

In [5]:
eligible_df = df.loc[~df["is_protected"]].copy().reset_index(drop=True)
assert len(eligible_df) >= 100, f"Need at least 100 eligible SKUs, found {len(eligible_df)}"

damaged = eligible_df["is_damaged"].astype(int).to_numpy()
dead = eligible_df["is_dead"].astype(int).to_numpy()
margin = eligible_df["effective_margin"].to_numpy(dtype=float)
volume = eligible_df["effective_volume"].to_numpy(dtype=float)

eligible_skus = set(eligible_df["SKU"])
sku_to_position = {sku: idx for idx, sku in enumerate(eligible_df["SKU"])}

dependency_edges = []
for _, row in eligible_df.iterrows():
    parent = str(row["Requires_Base_SKU"]).strip()
    if parent and parent in eligible_skus:
        dependency_edges.append((sku_to_position[parent], sku_to_position[row["SKU"]]))

print("Eligible SKUs:", len(eligible_df))
print("Dependency edges:", len(dependency_edges))

NameError: name 'df' is not defined

In [6]:
def solve_problem(target_count: int, cost_vector: np.ndarray, damaged_exact=None, dead_exact=None):
    rows = [np.ones(len(eligible_df))]
    lower = [target_count]
    upper = [target_count]

    if damaged_exact is not None:
        rows.append(damaged.astype(float))
        lower.append(damaged_exact)
        upper.append(damaged_exact)

    if dead_exact is not None:
        rows.append(dead.astype(float))
        lower.append(dead_exact)
        upper.append(dead_exact)

    for parent_idx, child_idx in dependency_edges:
        row = np.zeros(len(eligible_df))
        row[child_idx] = 1.0
        row[parent_idx] = -1.0
        rows.append(row)
        lower.append(-np.inf)
        upper.append(0.0)

    constraints = LinearConstraint(
        np.vstack(rows),
        np.array(lower),
        np.array(upper),
    )

    result = milp(
        c=cost_vector,
        constraints=constraints,
        integrality=np.ones(len(eligible_df), dtype=int),
        bounds=Bounds(np.zeros(len(eligible_df)), np.ones(len(eligible_df))),
    )

    if result.status != 0:
        raise RuntimeError(
            f"MILP failed for target={target_count}, damaged_exact={damaged_exact}, dead_exact={dead_exact}: {result.message}"
        )

    return result.x > 0.5


def solve_density_optimal(target_count: int):
    damaged_solution = solve_problem(target_count, -damaged.astype(float))
    optimal_damaged = int(damaged[damaged_solution].sum())

    dead_solution = solve_problem(target_count, -dead.astype(float), damaged_exact=optimal_damaged)
    optimal_dead = int(dead[dead_solution].sum())

    best_density = 0.0
    best_solution = None

    for _ in range(50):
        solution = solve_problem(
            target_count,
            margin - best_density * volume,
            damaged_exact=optimal_damaged,
            dead_exact=optimal_dead,
        )

        selected_margin = float(margin[solution].sum())
        selected_volume = float(volume[solution].sum())
        updated_density = selected_margin / selected_volume if selected_volume > 0 else 0.0

        best_solution = solution
        if abs(updated_density - best_density) < 1e-10:
            best_density = updated_density
            break
        best_density = updated_density

    selected = eligible_df.loc[best_solution].copy()
    return {
        "selected": selected,
        "optimal_damaged": optimal_damaged,
        "optimal_dead": optimal_dead,
        "density": best_density,
        "margin": float(selected["effective_margin"].sum()),
        "volume": float(selected["effective_volume"].sum()),
    }


def build_ranked_output(selected_df: pd.DataFrame) -> pd.DataFrame:
    ranked = selected_df.copy()

    ranked["priority_tier"] = np.select(
        [ranked["is_damaged"], ranked["is_dead"]],
        [0, 1],
        default=2,
    )
    ranked = ranked.sort_values(
        by=["priority_tier", "effective_volume", "SKU"],
        ascending=[True, False, True],
    ).reset_index(drop=True)
    ranked.insert(0, "Priority_Rank", np.arange(1, len(ranked) + 1))
    return ranked[["Priority_Rank", "SKU", "Product_Name"]]

## Solve both list sizes

In [7]:
results = {}

for target in (50, 100):
    result = solve_density_optimal(target)
    ranked_output = build_ranked_output(result["selected"])

    output_path = WORKSPACE_DIR / f"liquidation_list_{target}.csv"
    ranked_output.to_csv(output_path, index=False)

    results[target] = {
        **result,
        "ranked_output": ranked_output,
    }

    print(f"Target {target}")
    print("Damaged selected:", result["optimal_damaged"])
    print("Dead selected:", result["optimal_dead"])
    print("Density:", round(result["density"], 6))
    display(ranked_output.head(10))

NameError: name 'damaged' is not defined

## Validate outputs and save final report

In [8]:
for target in (50, 100):
    selected_skus = set(results[target]['selected']['SKU'])
    protected_skus = set(df.loc[df['is_protected'], 'SKU'])
    assert not (selected_skus & protected_skus), 'Protected SKU found in selection'
    assert len(selected_skus) == target, f'Expected {target} SKUs, got {len(selected_skus)}'

    # Dependency consistency check (CORRECT direction):
    # Business rule: if a PARENT is liquidated, ALL its non-protected dependents
    # must also be liquidated.  We scan the full df for rows whose
    # Requires_Base_SKU appears in selected_skus; those child SKUs must be
    # selected too (unless they are protected and therefore ineligible).
    for _, row in df.iterrows():
        parent = row['Requires_Base_SKU']
        if pd.notna(parent):
            parent = str(parent).strip()
            if parent and parent in selected_skus:
                child_sku = row['SKU']
                if not row['is_protected']:
                    assert child_sku in selected_skus, (
                        f'Dependency violated for target={target}: '
                        f'parent {parent} is selected but dependent {child_sku} is not'
                    )
    print(f'target={target}: all checks passed')

final_report = pd.DataFrame([{
    'reclaimed_space_100': results[100]['volume'],
    'total_margin_density_100': results[100]['density'],
    'total_margin_100': results[100]['margin'],
    'reclaimed_space_50': results[50]['volume'],
    'total_margin_density_50': results[50]['density'],
    'total_margin_50': results[50]['margin'],
}])

final_report_path = WORKSPACE_DIR / 'final_report.csv'
final_report.to_csv(final_report_path, index=False)

print('Saved:')
print(WORKSPACE_DIR / 'liquidation_list_50.csv')
print(WORKSPACE_DIR / 'liquidation_list_100.csv')
print(final_report_path)
display(final_report)

KeyError: 50